# Leitura de dados brutos, CSV e JSON

Neste notebook, vamos conhecer diferentes formas de trazer dados
- lendo bytes e texto diretamente
- interpretando arquivos CSV e json com a biblioteca padrão e usando **pandas**

## Objetivos
- Diferenciar a leitura de bytes da interpretação dos dados.
- Ler um arquivo texto e um CSV com ferramentas da biblioteca padrão.
- Ler CSV e JSON com pandas e inspecionar os dados.
- Reconhecer problemas comuns de separador, codificação, tipos e dados ausentes.
- Fazer requisições HTTP E percorrer páginas.
- Extrair e relacionar dados de um banco OLTP local usando SQLite.

### Observações
- Os exemplos de arquivos e do banco SQLite serão salvos na pasta `dados_aula`.
- Execute cada bloco de código em sequência para que tudo funcione.

## 1. Preparação

`pathlib` facilita a criação de caminhos de arquivos. `csv` e `json` fazem parte da biblioteca padrão do Python, portanto não precisam ser instalados. O pandas será importado mais adiante.

In [1]:
from pathlib import Path
import csv
import json

pasta_dados = Path("dados_aula")
pasta_dados.mkdir(exist_ok=True)
print(f"Arquivos de demonstração serão salvos em: {pasta_dados.resolve()}")

Arquivos de demonstração serão salvos em: C:\Users\ajb95\Faculdade\LADATA\SEMAC\testes\dados_aula


## 2. Leitura de dados brutos

“Bruto” significa que ainda não interpretamos o conteúdo como uma tabela ou estrutura específica. Arquivos podem ser lidos como **texto** (caracteres) ou como **bytes** (valores binários). Texto exige atenção à codificação, como UTF-8. Bytes são úteis para arquivos binários, como imagens, áudio e formatos compactados.

In [2]:
arquivo_texto = pasta_dados / "anotacoes.txt"
arquivo_texto.write_text("Amostra de dados brutos.\nTemperatura: 23.5 °C\n", encoding="utf-8")

# Leitura como texto: o arquivo é aberto para leitura e tem os bytes decodificados usando UTF-8.
with arquivo_texto.open("r", encoding="utf-8") as arquivo:
    conteudo_texto = arquivo.read()
print(conteudo_texto)

# Leitura binária: recebemos bytes, sem interpretação automática como texto.
conteudo_bytes = arquivo_texto.read_bytes()
print("Tipo:", type(conteudo_bytes))
print("Primeiros bytes:", conteudo_bytes[:30])
print("Bytes em hexadecimal:", conteudo_bytes[:12].hex())

Amostra de dados brutos.
Temperatura: 23.5 °C

Tipo: <class 'bytes'>
Primeiros bytes: b'Amostra de dados brutos.\r\nTemp'
Bytes em hexadecimal: 416d6f737472612064652064


O mesmo arquivo pode ser visto como texto ou como bytes. Ao ler bytes de um arquivo que sabemos ser texto, podemos decodificá-los explicitamente com `.decode("utf-8")`. Para dados binários, não devemos tentar decodificar sem saber o formato.

In [3]:
texto_decodificado = conteudo_bytes.decode("utf-8")
print(texto_decodificado)

Amostra de dados brutos.
Temperatura: 23.5 °C



### Exercício 1 !

A loja virtual em que você trabalha recebeu um arquivo de registro de atividades do sistema. Antes de iniciar o processamento, a equipe precisa inspecionar o arquivo para identificar seu conteúdo. 

- Leia o arquivo  chamado log_sistema.txt utilizando Python. Ele esta na pasta exercicios/modulo01
- Exiba o conteúdo completo.
- Leia o mesmo arquivo como bytes e apresente os dez primeiros bytes.

In [1]:
# escreva seu código aqui

## 3. CSV com a biblioteca padrão (`csv`)

CSV (comma separated values) representa linhas e colunas em texto separado por um delimitador. Embora a vírgula seja comum, arquivos também podem usar ponto e vírgula. `csv.DictReader` transforma cada linha em um dicionário cujas chaves são os nomes das colunas. Sem conversão adicional, os valores lidos são textos.

In [4]:
#guardando valores num arquivo csv
arquivo_csv = pasta_dados / "alunos.csv" # cria o arquivo na pasta
linhas_alunos = [
    {"nome": "Ana", "idade": 20, "nota": 8.5},
    {"nome": "Bruno", "idade": 22, "nota": 7.0},
    {"nome": "Caio", "idade": 19, "nota": 9.2},
] # dicionario com valores representado nome, idade e nota, que serão o cabeçalho da nossa tabela

with arquivo_csv.open("w", newline="", encoding="utf-8") as arquivo: #abertura do arquivo para escrita
    escritor = csv.DictWriter(arquivo, fieldnames=["nome", "idade", "nota"]) #declarando os nomes das colunas (cabeçalho)
    escritor.writeheader() # escrita do cabeçalho no csv
    escritor.writerows(linhas_alunos) #escrita de cada linha e seus valores nas respectivas colunas

#leitura do que foi escrito no arquivo
with arquivo_csv.open("r", newline="", encoding="utf-8") as arquivo:
    leitor = csv.DictReader(arquivo)
    alunos = list(leitor) # guarda o conteudo na lista de alunos

print(alunos)
print("Tipo da idade lida:", type(alunos[0]["idade"]))

[{'nome': 'Ana', 'idade': '20', 'nota': '8.5'}, {'nome': 'Bruno', 'idade': '22', 'nota': '7.0'}, {'nome': 'Caio', 'idade': '19', 'nota': '9.2'}]
Tipo da idade lida: <class 'str'>


## 4. CSV com pandas

O pandas representa dados tabulares em um `DataFrame`, facilitando filtros, conversões, estatísticas e exploração. Se o import abaixo falhar porque pandas não está instalado, adicione uma célula de código e execute: `%pip install pandas`, depois, execute novamente o import.

In [5]:
import pandas as pd

print("Versão do pandas:", pd.__version__)

Versão do pandas: 2.3.3


A função `read_csv` lê o arquivo e tenta inferir os tipos das colunas. Vamos criar uma tabela com um valor ausente e explorar suas primeiras linhas, tipos e estatísticas.

In [ ]:
#criando valores e gravando no arquivo vendas.csv
arquivo_vendas = pasta_dados / "vendas.csv"
vendas_exemplo = pd.DataFrame(
    {
        "produto": ["Café", "Chá", "Café", "Suco"],
        "quantidade": [3, 5, 2, 4],
        "preco_unitario": [12.5, 8.0, 12.5, 6.5],
        "loja": ["Centro", "Norte", "Centro", None],
    }
)
#index=false indica que nesse caso nao queremos que seja gravada uma coluna adicional com o indice de cada linha
vendas_exemplo.to_csv(arquivo_vendas, index=False, encoding="utf-8") 

#lendo o arquivo com pandas
vendas = pd.read_csv(arquivo_vendas, encoding="utf-8")

# head mostra as primeiras linhas do arquivo
display(vendas.head()) #
print("\nResumo das colunas e tipos:")
vendas.info()
print("\nEstatísticas numéricas:")
display(vendas.describe())
print("\nValores ausentes por coluna:")
display(vendas.isna().sum())

,produto,quantidade,preco_unitario,loja
0,Café,3,12.5,Centro
1,Chá,5,8.0,Norte
2,Café,2,12.5,Centro
3,Suco,4,6.5,NaN



Resumo das colunas e tipos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   produto         4 non-null      object 
 1   quantidade      4 non-null      int64  
 2   preco_unitario  4 non-null      float64
 3   loja            3 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 260.0+ bytes

Estatísticas numéricas:


,quantidade,preco_unitario
count,4.000000,4.000000
mean,3.500000,9.875000
std,1.290994,3.092329
min,2.000000,6.500000
25%,2.750000,7.625000
50%,3.500000,10.250000
75%,4.250000,12.500000
max,5.000000,12.500000



Valores ausentes por coluna:


produto           0
quantidade        0
preco_unitario    0
loja              1
dtype: int64

### Opções úteis ao ler CSV

- `sep`: delimitador, por exemplo `sep=";"`.
- `encoding`: codificação do arquivo, frequentemente `"utf-8"` ou `"latin-1"`.
- `usecols`: seleciona apenas algumas colunas.
- `dtype`: define tipos quando a inferência automática não é adequada.
- `na_values`: informa textos que devem ser tratados como valores ausentes.

Um separador ou encoding incorreto pode fazer a leitura falhar ou produzir colunas inesperadas, por isso é importante conhecer como o arquivo foi gerado e sua estrutura

In [7]:
arquivo_ponto_virgula = pasta_dados / "estoque.csv"
arquivo_ponto_virgula.write_text(
    "item;quantidade\nCaderno;10\nCaneta;25\n", encoding="utf-8"
)

estoque = pd.read_csv(arquivo_ponto_virgula, sep=";", encoding="utf-8")
display(estoque)
print("Tipos inferidos:", estoque.dtypes.to_dict())

,item,quantidade
0,Caderno,10
1,Caneta,25


Tipos inferidos: {'item': dtype('O'), 'quantidade': dtype('int64')}


### Exercício 2 !!

O departamento comercial enviou um arquivo com os produtos ofertados pela loja. Entretanto, você desconfia que a pessoa que preencheu as informações deixou campos ausentes, sua equipe precisa entender a qualidade dos dados antes de passa-los a diante.

Utilizando o arquivo de vendas disponivel em exercicios/modulo01/produtos.csv
1. Carregue o CSV em um DataFrame.
2. Exiba as cinco primeiras vendas.
3. Identifique os tipos de dados de cada coluna.
4. Identifique quantos valores ausentes existem em cada coluna.

In [ ]:
# Seu codigo aqui

## 5. JSON com a biblioteca padrão (`json`)

JSON é um formato textual que representa objetos, listas, números, textos, valores booleanos e `null`. Em Python, `json.load` lê JSON de um arquivo aberto; `json.loads` interpreta uma string. O resultado costuma ser um dicionário ou uma lista.

In [8]:
arquivo_json = pasta_dados / "aluno.json"
aluno = {
    "nome": "Lia",
    "curso": "Ciência de Dados",
    "ativo": True,
    "notas": [8.5, 9.0],
}

with arquivo_json.open("w", encoding="utf-8") as arquivo:
    json.dump(aluno, arquivo, ensure_ascii=False, indent=2)

with arquivo_json.open("r", encoding="utf-8") as arquivo:
    aluno_lido = json.load(arquivo)

print(aluno_lido)
print("Nome:", aluno_lido["nome"])
print("Média das notas:", sum(aluno_lido["notas"]) / len(aluno_lido["notas"]))

# Também podemos interpretar JSON que já está em uma string.
texto_json = '[{"cidade": "Recife", "temperatura": 28}, {"cidade": "Curitiba", "temperatura": 17}]'
cidades = json.loads(texto_json)
print("\nLista interpretada a partir de uma string:", cidades)

{'nome': 'Lia', 'curso': 'Ciência de Dados', 'ativo': True, 'notas': [8.5, 9.0]}
Nome: Lia
Média das notas: 8.75

Lista interpretada a partir de uma string: [{'cidade': 'Recife', 'temperatura': 28}, {'cidade': 'Curitiba', 'temperatura': 17}]


## 6. JSON com pandas

Para JSON organizado como uma lista de objetos com campos semelhantes, `pd.read_json` pode criar um `DataFrame` diretamente. Para respostas de APIs ou estruturas aninhadas, `pd.json_normalize` ajuda a transformar campos internos em colunas.

In [9]:
arquivo_cidades = pasta_dados / "cidades.json"
registros_cidades = [
    {"cidade": "Recife", "temperatura": 28, "estado": "PE"},
    {"cidade": "Curitiba", "temperatura": 17, "estado": "PR"},
    {"cidade": "Manaus", "temperatura": 31, "estado": "AM"},
]
with arquivo_cidades.open("w", encoding="utf-8") as arquivo:
    json.dump(registros_cidades, arquivo, ensure_ascii=False, indent=2)

cidades_df = pd.read_json(arquivo_cidades, encoding="utf-8")
display(cidades_df)
print("Temperatura média:", cidades_df["temperatura"].mean())

,cidade,temperatura,estado
0,Recife,28,PE
1,Curitiba,17,PR
2,Manaus,31,AM


Temperatura média: 25.333333333333332


In [10]:
resposta_api = {
    "fonte": "estacao_local",
    "leituras": [
        {"local": {"cidade": "Recife", "estado": "PE"}, "medicao": {"temperatura": 28, "umidade": 70}},
        {"local": {"cidade": "Curitiba", "estado": "PR"}, "medicao": {"temperatura": 17, "umidade": 55}},
    ],
}

# record_path escolhe a lista de registros; meta preserva um campo do objeto externo.
leituras_df = pd.json_normalize(
    resposta_api,
    record_path="leituras",
    meta=["fonte"],
    sep="_",
)
display(leituras_df)

,local_cidade,local_estado,medicao_temperatura,medicao_umidade,fonte
0,Recife,PE,28,70,estacao_local
1,Curitiba,PR,17,55,estacao_local


## 7. Qual ferramenta escolher?

| Situação | Ferramenta indicada |
| --- | --- |
| Ler bytes de um arquivo ou texto simples | `Path.read_bytes()` / `open()` |
| Processar CSV pequeno sem dependências extras | `csv.DictReader` |
| Explorar, filtrar e analisar dados tabulares | `pandas.read_csv` |
| Interpretar JSON e preservar sua estrutura | `json.load` / `json.loads` |
| Converter JSON tabular ou aninhado em colunas | `pandas.read_json` / `pandas.json_normalize` |

### Boas práticas
1. Confira a codificação e o delimitador do arquivo.
2. Inspecione as primeiras linhas e os tipos antes de analisar.
3. Verifique valores ausentes e converta tipos quando necessário.
4. Use `with open(...)` para garantir o fechamento dos arquivos.
5. Para arquivos grandes, considere processar em partes (por exemplo, `chunksize` no `read_csv`).


**Observação:** os arquivos de exemplo são recriados quando as células correspondentes são executadas. Execute as células de cima para baixo para acompanhar a aula.

### Exercicio 3 !!!

Uma API interna fornece informações sobre pedidos, mas a estrutura recebida contém informações aninhadas, dificultando a análise.

Considere a seguinte estrutura de dados:

In [42]:
resposta_api = {
    "loja": "Loja Virtual",
    "pedidos": [
        {
            "id": 101,
            "cliente": {
                "nome": "Ana",
                "cidade": "Aracaju"
            },
            "valor": 250.00
        },
        {
            "id": 102,
            "cliente": {
                "nome": "Bruno",
                "cidade": "Recife"
            },
            "valor": 180.00
        },
        {
            "id": 103,
            "cliente": {
                "nome": "Carla",
                "cidade": "Aracaju"
            },
            "valor": 420.00
        }
    ]
}

- Transforme a lista de pedidos em um DataFrame.
- Faça com que as informações de cliente também sejam apresentadas como colunas.
- Calcule o faturamento total dos pedidos.
- Mostre apenas os pedidos realizados por clientes de Aracaju.

Observação: lembre de executar a celula de codigo com a estrutura json

In [ ]:
# Escreva seu codigo aqui

## 9. Ingestão via API REST

Uma API REST funciona como uma ponte de comunicação para extrair dados da web. Em um processo clássico de ingestão, o engenheiro de dados faz um pedido à API (requisição GET), verifica se o status da resposta foi um sucesso, interpreta os dados recebidos (geralmente em formato JSON) e percorre as páginas seguintes até extrair tudo o que precisa.
No mundo corporativo, essas APIs costumam exigir autenticação, impor limites de acesso rápido e usar regras próprias de paginação. Então, para simularmos esses desafios reais de forma prática, vamos consumir uma API pública de verdade: a PokeAPI. 

- https://pokeapi.co/

No código a seguir, vamos demontrar passo a passo como se dá todo esse processo.


In [44]:
import urllib.request # Biblioteca nativa do Python para fazer requisições web
import json           # Biblioteca para transformar o texto da web em dicionários do Python
import time           # Importamos a biblioteca 'time' para dar uma pausa entre as requisições

# URL de uma API pública. Escolhemos a de Pokémon!
# O parâmetro '?limit=6' diz para a API nos entregar apenas 6 resultados por vez em cada ágina
url = "https://pokeapi.co/api/v2/pokemon?limit=6"

# Precisamos criar um "disfarce" dizendo que somos um navegador (como o Chrome/Mozilla).
# Isso é necessário pois muitos servidores bloqueiam acessos que parecem ser de "robôs"
cabecalho = {'User-Agent': 'Mozilla/5.0'}

pagina_atual = 0
limite_paginas = 5  # Vamos extrair apenas 5 páginas como exemplo
   
# Precisamos de um while para que todas as páginas sejam buscadas seguindo o mesmo ciclo
#Equanto existir uma URL válida E não atingirmos o limite de páginas
   
while url and pagina_atual <= limite_paginas:
    print(f"\n--- Extraindo Página {pagina_atual} ---")

    # 1. Fazer a requisição GET passando a URL e o disfarce
    requisicao = urllib.request.Request(url, headers=cabecalho, method="GET")
    
    # A estrutura try-except é fundamental na Engenharia de Dados para tratar erros 
    # de conexão (ex: internet cair) sem que o programa quebre 
    try:
        with urllib.request.urlopen(requisicao) as resposta:
            # 2. Verificando o status da resposta. 200 é a resposta que indica que a conexão foi um sucesso
            print(f"Status da resposta: {resposta.status}")
            
            if resposta.status == 200:
                # lendo o conteúdo e decodificando para um texto legível
                conteudo = resposta.read().decode("utf-8")
                
                # 3. Transformamos o texto puro em um formato estruturado ->JSON
                dados_api = json.loads(conteudo)
                
                # Exibimos o total apenas na primeira página para não repetir nos prints seguintes
                if pagina_atual == 0:
                    print(f"Total de recursos disponíveis: {dados_api['count']}")
                
                print("Resultados desta página:")
                
                # Usamos o laço for para percorrer a lista de resultados da página
                for item in dados_api['results']:
                    print(f"- {item['name'].capitalize()}")#tratamento mostrar os dados .capitalize()
                    
                
                #Substituímos a nossa URL antiga pelo link da PRÓXIMA página!
                url = dados_api['next']
                
        # Atualizamos o contador de páginas para o loop não rodar infinitamente
        pagina_atual += 1
        
        #Pausar 1 segundo antes de pedir a próxima página para não sobrecarregar o servidor.
        time.sleep(1)

    except urllib.error.URLError as e:#qualquer erro inesperado cái nessa exeção, importante para identificarmos facilmente o erro.
        print(f"Falha na requisição: {e.reason}")
        break # Interrompe o loop em caso de erro


--- Extraindo Página 0 ---
Status da resposta: 200
Total de recursos disponíveis: 1351
Resultados desta página:
- Bulbasaur
- Ivysaur
- Venusaur
- Charmander
- Charmeleon
- Charizard

--- Extraindo Página 1 ---
Status da resposta: 200
Resultados desta página:
- Squirtle
- Wartortle
- Blastoise
- Caterpie
- Metapod
- Butterfree

--- Extraindo Página 2 ---
Status da resposta: 200
Resultados desta página:
- Weedle
- Kakuna
- Beedrill
- Pidgey
- Pidgeotto
- Pidgeot

--- Extraindo Página 3 ---
Status da resposta: 200
Resultados desta página:
- Rattata
- Raticate
- Spearow
- Fearow
- Ekans
- Arbok

--- Extraindo Página 4 ---
Status da resposta: 200
Resultados desta página:
- Pikachu
- Raichu
- Sandshrew
- Sandslash
- Nidoran-f
- Nidorina

--- Extraindo Página 5 ---
Status da resposta: 200
Resultados desta página:
- Nidoqueen
- Nidoran-m
- Nidorino
- Nidoking
- Clefairy
- Clefable


**Códigos de Status HTTP**

Toda vez que você faz uma requisição a uma API, o servidor responde com um código numérico de três dígitos. Na Engenharia de Dados, monitorar esses números é importante para saber se a ingestão funcionou ou se o seu código precisa de alguma ação.

Aqui estão os principais códigos que você vai encontrar no dia a dia:

- 200 OK: É o cenário ideal. Significa que a requisição foi um sucesso e os dados foram entregues perfeitamente.

- 400 Bad Request: O servidor não entendeu o seu pedido. Geralmente ocorre por um erro de digitação na URL ou parâmetros incorretos no código.

- 401 Unauthorized / 403 Forbidden: Problema de permissão. O servidor bloqueou o seu acesso porque faltou uma senha (token) ou porque identificou o seu código como um robô.

- 404 Not Found: O servidor está no ar, mas a informação solicitada não existe.

- 429 Too Many Requests: Limite de velocidade excedido. Você está enviando requisições rápido demais e o servidor bloqueou temporariamente para não sobrecarregar.

- 500 Internal Server Error: O servidor de destino falhou ou caiu.

### Exercicio 4

A loja deseja enriquecer seu catálogo com informações de uma fonte externa. A equipe recebeu a tarefa de coletar dados de uma API pública.

Utilizando a PokeAPI apresentada na seção:
- Realize uma requisição para obter informações de Pokémon.
- Altere a quantidade de resultados solicitados por página.
- Percorra as páginas utilizando a URL de próxima página retornada pela API.
- Armazene os nomes e as URLs dos Pokémon encontrados em uma lista.
- Transforme os resultados em um DataFrame e o exiba.
- Salve os resultados num csv


In [ ]:
#Seu codigo aqui

## 10. Ingestão de banco de dados relacional (OLTP)

Bancos OLTP (*Online Transaction Processing*) são voltados a operações do dia a dia, como cadastrar clientes e registrar pedidos. Os dados costumam estar organizados em tabelas relacionadas. Para simular a extração sem instalar um servidor, usaremos o SQLite local e o módulo `sqlite3`, incluído no Python.

In [61]:
import sqlite3

arquivo_banco = pasta_dados / "loja_oltp.db"
conexao = sqlite3.connect(arquivo_banco)

try:
    # Cria duas tabelas relacionadas.
    conexao.execute("""
        CREATE TABLE clientes (
            cliente_id INTEGER PRIMARY KEY,
            nome TEXT NOT NULL,
            cidade TEXT NOT NULL
        )
    """)

    conexao.execute("""
        CREATE TABLE pedidos (
            pedido_id INTEGER PRIMARY KEY,
            cliente_id INTEGER NOT NULL,
            data_pedido TEXT NOT NULL,
            valor REAL NOT NULL,
            FOREIGN KEY (cliente_id) REFERENCES clientes(cliente_id)
        )
    """)

    # Insere alguns registros fictícios.
    conexao.executemany(
        "INSERT INTO clientes (cliente_id, nome, cidade) VALUES (?, ?, ?)",
        [
            (1, "Ana", "Recife"),
            (2, "Bruno", "Curitiba"),
            (3, "Lia", "Manaus"),
        ],
    )

    conexao.executemany(
        """INSERT INTO pedidos
           (pedido_id, cliente_id, data_pedido, valor)
           VALUES (?, ?, ?, ?)""",
        [
            (1001, 1, "2026-03-01", 45.90),
            (1002, 2, "2026-03-02", 18.00),
            (1003, 1, "2026-03-03", 12.50),
            (1004, 3, "2026-03-04", 75.00),
            (1005, 2, "2026-03-05", 32.25),
        ],
    )

    conexao.commit()

finally:
    conexao.close()

print("Banco de demonstração criado em:", arquivo_banco)

Banco de demonstração criado em: dados_aula\loja_oltp.db


### Conectar e executar uma consulta
A função sqlite3.connect() abre a conexão com o arquivo do banco. Com cursor.execute(), executamos uma instrução SQL. O método fetchall() recupera todas as linhas retornadas pela consulta.

Neste primeiro exemplo, vamos consultar os pedidos sem relacioná-los a outras tabelas.

In [62]:
conexao = sqlite3.connect(arquivo_banco)

try:
    cursor = conexao.cursor()

    cursor.execute("""
        SELECT pedido_id, data_pedido, valor
        FROM pedidos
        ORDER BY pedido_id
    """)

    registros = cursor.fetchall()

    for registro in registros:
        print(registro)

finally:
    cursor.close()
    conexao.close()

(1001, '2026-03-01', 45.9)
(1002, '2026-03-02', 18.0)
(1003, '2026-03-03', 12.5)
(1004, '2026-03-04', 75.0)
(1005, '2026-03-05', 32.25)


O resultado de fetchall() é uma lista de tuplas. Quando não precisamos de todos os registros, podemos recuperar apenas um (fetchone()) ou uma quantidade limitada (fetchmany()).

Também podemos filtrar os dados diretamente no SQL. Para isso, utilizamos parâmetros (?) em vez de inserir valores diretamente no texto da consulta. Essa prática evita problemas com valores especiais e reduz riscos de injeção de SQL.

In [63]:
conexao = sqlite3.connect(arquivo_banco)

try:
    cursor = conexao.cursor()

    consulta = """
        SELECT pedido_id, data_pedido, valor
        FROM pedidos
        WHERE valor >= ?
        ORDER BY valor DESC
    """

    valor_minimo = 30.00
    cursor.execute(consulta, (valor_minimo,))

    print("Pedidos com valor igual ou superior a R$ 30:")
    for pedido in cursor.fetchall():
        print(pedido)

finally:
    cursor.close()
    conexao.close()

Pedidos com valor igual ou superior a R$ 30:
(1004, '2026-03-04', 75.0)
(1001, '2026-03-01', 45.9)
(1005, '2026-03-05', 32.25)


Em bancos relacionais, uma informação pode estar distribuída em diferentes tabelas. O JOIN permite combinar registros relacionados.

No exemplo, a tabela pedidos armazena o identificador do cliente, enquanto o nome e a cidade estão na tabela clientes. A consulta reúne essas informações em um único resultado.

In [64]:
conexao = sqlite3.connect(arquivo_banco)

try:
    consulta = """
        SELECT
            pedidos.pedido_id,
            pedidos.data_pedido,
            clientes.nome AS cliente,
            clientes.cidade,
            pedidos.valor
        FROM pedidos
        JOIN clientes
            ON pedidos.cliente_id = clientes.cliente_id
        ORDER BY pedidos.pedido_id
    """

    cursor = conexao.cursor()
    cursor.execute(consulta)

    for pedido in cursor.fetchall():
        print(pedido)

finally:
    cursor.close()
    conexao.close()

(1001, '2026-03-01', 'Ana', 'Recife', 45.9)
(1002, '2026-03-02', 'Bruno', 'Curitiba', 18.0)
(1003, '2026-03-03', 'Ana', 'Recife', 12.5)
(1004, '2026-03-04', 'Lia', 'Manaus', 75.0)
(1005, '2026-03-05', 'Bruno', 'Curitiba', 32.25)


Além de recuperar os registros com sqlite3, podemos utilizar o pandas para executar uma consulta e receber o resultado diretamente em um DataFrame.

Essa integração é útil quando os dados extraídos serão explorados, analisados ou encaminhados para outras etapas do fluxo de engenharia de dados.

In [65]:
conexao = sqlite3.connect(arquivo_banco)

try:
    pedidos_df = pd.read_sql_query(consulta, conexao)
    display(pedidos_df)

finally:
    conexao.close()

,pedido_id,data_pedido,cliente,cidade,valor
0,1001,2026-03-01,Ana,Recife,45.90
1,1002,2026-03-02,Bruno,Curitiba,18.00
2,1003,2026-03-03,Ana,Recife,12.50
3,1004,2026-03-04,Lia,Manaus,75.00
4,1005,2026-03-05,Bruno,Curitiba,32.25


### Extrair dados em lotes
Quando uma consulta retorna muitos registros, carregar tudo de uma vez pode consumir muita memória. O método fetchmany(tamanho) permite recuperar os resultados em grupos menores.

Neste exemplo, cada chamada recupera até dois registros. Em um fluxo real, os lotes poderiam ser enviados para arquivos, outro banco de dados ou uma etapa de processamento.

In [66]:
conexao = sqlite3.connect(arquivo_banco)

try:
    cursor = conexao.cursor()
    cursor.execute(consulta)

    tamanho_lote = 2
    numero_lote = 1

    while True:
        lote = cursor.fetchmany(tamanho_lote)

        if not lote:
            break

        print(f"Lote {numero_lote}:")
        for pedido in lote:
            print(pedido)

        numero_lote += 1

finally:
    cursor.close()
    conexao.close()

Lote 1:
(1001, '2026-03-01', 'Ana', 'Recife', 45.9)
(1002, '2026-03-02', 'Bruno', 'Curitiba', 18.0)
Lote 2:
(1003, '2026-03-03', 'Ana', 'Recife', 12.5)
(1004, '2026-03-04', 'Lia', 'Manaus', 75.0)
Lote 3:
(1005, '2026-03-05', 'Bruno', 'Curitiba', 32.25)


### Cuidados ao extrair dados relacionais

- Conheça a estrutura: identifique tabelas, colunas, chaves e relacionamentos antes de escrever consultas.
- Selecione o necessário: evite trazer todas as colunas e todos os registros quando isso não for necessário.
- Use parâmetros: passe valores por meio de ?, sem concatenar dados diretamente no comando SQL.
- Considere o volume: fetchmany() permite ler o resultado em lotes; para volumes maiores, também podem ser necessárias estratégias de paginação ou extração incremental.
- Evite sobrecarregar a origem: consultas muito longas ou frequentes podem afetar o banco utilizado pelos sistemas operacionais.
- Feche a conexão: utilize try/finally ou outra estrutura que garanta o fechamento após a extração.


### Exercicio 4 !!!!

Você voltou de ferias enquanto brincava com APIs e a loja agora utiliza um sistema OLTP para registrar clientes e pedidos. A equipe de engenharia de dados precisa extrair informações para alimentar o ambiente analítico.


Utilizando o banco SQLite criado na seção:
- Consulte todos os clientes cadastrados.
- Consulte os pedidos com valor superior a R$ 30.
- Recupere os registros em lotes de dois utilizando fetchmany()
- Carregue o resultado da consulta com Pandas em um DataFrame e o exiba.

## Gabarito dos Exercícios

### Exercicio 1 - escrita e leitura de dados brutos

In [7]:
from pathlib import Path

#Criar o caminho para o arquivo usando Path()
caminho_log = Path("exercicios/modulo01/log_sistema.txt")


#Ler o arquivo como texto e exibir o conteúdo completo usando read_text()
print("Conteudo em texto: ")
conteudo_texto = caminho_log.read_text(encoding="utf-8")
print(conteudo_texto)

#Ler o conteúdo como bytes usando read_bytes() e pegar os 10 primeiros
print("Leitura em bytes e exibição dos 20 primeiros: ")
conteudo_bytes = caminho_log.read_bytes()
dez_primeiros_bytes = conteudo_bytes[:20]
print(f"10 primeiros bytes: {dez_primeiros_bytes}")

#Converter os 10 primeiros bytes em texto usando decode()
texto_decodificado = dez_primeiros_bytes.decode("utf-8")
print(f"10 primeiros bytes convertidos em texto: {texto_decodificado}")



Conteudo em texto: 
[2026-11-25 08:00:01] INFO: Sistema da loja iniciado.
[2026-11-25 08:05:12] LOGIN: Usuario 'carlos_silva' entrou na conta.
[2026-11-25 08:07:45] CARRINHO: Item 'Teclado Mecanico' adicionado.
[2026-11-25 08:10:03] PAGAMENTO: Pedido #4092 aprovado via PIX.
[2026-11-25 08:12:30] ERRO: Falha ao calcular frete para CEP 49000-000.

Leitura em bytes e exibição dos 20 primeiros: 
10 primeiros bytes: b'[2026-11-25 08:00:01'
10 primeiros bytes convertidos em texto: [2026-11-25 08:00:01


### Exercício 2 - leitura csv

In [40]:
import pandas as pd

#Carregue o CSV em um DataFrame
df_vendas = pd.read_csv("exercicios/modulo01/produtos.csv")

#Exiba as cinco primeiras vendas
display(df_vendas.head())

#Identifique os tipos de dados de cada coluna
print("\nEstrutura e tipos de dados")
df_vendas.info()

#Identifique quantos valores ausentes existem em cada coluna
print("\nValores ausentes por coluna")
print(df_vendas.isna().sum())

,id_produto,nome,categoria,preco
0,1,Mouse Sem Fio,Perifericos,85.9
1,2,Teclado Mecanico,Perifericos,249.9
2,3,Cabo HDMI 2m,Acessorios,35.0
3,4,Monitor 24 Polegadas,Monitores,799.0
4,5,Headset Gamer,Audio,159.5



Estrutura e tipos de dados
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id_produto  25 non-null     int64  
 1   nome        24 non-null     object 
 2   categoria   23 non-null     object 
 3   preco       23 non-null     float64
dtypes: float64(1), int64(1), object(2)
memory usage: 932.0+ bytes

Valores ausentes por coluna
id_produto    0
nome          1
categoria     2
preco         2
dtype: int64


In [41]:
# Mostrando como seria excluir os valores ausentes 
df_vendas_sem_nulos = df_vendas.dropna()
print("\nNova checagem valores ausentes por coluna")
print(df_vendas_sem_nulos.isna().sum())


Nova checagem valores ausentes por coluna
id_produto    0
nome          0
categoria     0
preco         0
dtype: int64


### Exercicio 3 - JSON

In [43]:
# Transformar a lista de pedidos em DataFrame achatando os dados do cliente em colunas
df_pedidos = pd.json_normalize(resposta_api["pedidos"])

print("Pedidos como dataframe")
display(df_pedidos)

# Calcular o faturamento total dos pedidos
faturamento_total = df_pedidos["valor"].sum()
print(f"Total: R$ {faturamento_total:.2f}")

#Identificar quais pedidos foram realizados por clientes de Aracaju
pedidos_aracaju = df_pedidos.loc[df_pedidos["cliente.cidade"] == "Aracaju"]
display(pedidos_aracaju)

# (Bônus usando groupby) Faturamento agrupado por cidade
print("\nFaturamento por cidade")
print(df_pedidos.groupby("cliente.cidade")["valor"].sum())

Pedidos como dataframe


,id,valor,cliente.nome,cliente.cidade
0,101,250.0,Ana,Aracaju
1,102,180.0,Bruno,Recife
2,103,420.0,Carla,Aracaju


Total: R$ 850.00


,id,valor,cliente.nome,cliente.cidade
0,101,250.0,Ana,Aracaju
2,103,420.0,Carla,Aracaju



Faturamento por cidade
cliente.cidade
Aracaju    670.0
Recife     180.0
Name: valor, dtype: float64


### Exercicio 4 - API

In [49]:
url = "https://pokeapi.co/api/v2/pokemon?limit=6"

cabecalho = {'User-Agent': 'Mozilla/5.0'}

pagina_atual = 0
limite_paginas = 7
dados_coletados = []
   
# Precisamos de um while para que todas as páginas sejam buscadas seguindo o mesmo ciclo
#Equanto existir uma URL válida E não atingirmos o limite de páginas
   
while url and pagina_atual <= limite_paginas:
    print(f"--- Extraindo Página {pagina_atual} ---")

    requisicao = urllib.request.Request(url, headers=cabecalho, method="GET")

    try:
        with urllib.request.urlopen(requisicao) as resposta:
            
            if resposta.status == 200:
                conteudo = resposta.read().decode("utf-8")
                dados_api = json.loads(conteudo)
                
                for item in dados_api['results']:
                    # Adicionado para salvar nome e url na lista
                    dados_coletados.append({
                        "nome": item["name"].capitalize(),
                        "url": item["url"]
                    })
                
                url = dados_api['next']
                
        pagina_atual += 1
        
        time.sleep(1)

    except urllib.error.URLError as e:
        print(f"Falha na requisição: {e.reason}")
        break

df_pokemon = pd.DataFrame(dados_coletados)
display(df_pokemon)

df_pokemon.to_csv("exercicios/modulo01/pokemon_exercicio.csv", index=False, encoding="utf-8")

--- Extraindo Página 0 ---
--- Extraindo Página 1 ---
--- Extraindo Página 2 ---
--- Extraindo Página 3 ---
--- Extraindo Página 4 ---
--- Extraindo Página 5 ---
--- Extraindo Página 6 ---
--- Extraindo Página 7 ---


,nome,url
0,Bulbasaur,https://pokeapi.co/api/v2/pokemon/1/
1,Ivysaur,https://pokeapi.co/api/v2/pokemon/2/
2,Venusaur,https://pokeapi.co/api/v2/pokemon/3/
3,Charmander,https://pokeapi.co/api/v2/pokemon/4/
4,Charmeleon,https://pokeapi.co/api/v2/pokemon/5/
5,Charizard,https://pokeapi.co/api/v2/pokemon/6/
6,Squirtle,https://pokeapi.co/api/v2/pokemon/7/
7,Wartortle,https://pokeapi.co/api/v2/pokemon/8/
8,Blastoise,https://pokeapi.co/api/v2/pokemon/9/
9,Caterpie,https://pokeapi.co/api/v2/pokemon/10/


### Exercicio 5 - OLTP

In [69]:
arquivo_banco = "exercicios/modulo01/loja_oltp.db"

# Conecta ao banco de dados existente
conexao = sqlite3.connect(arquivo_banco)
cursor = conexao.cursor()

try:
    # Consultar todos os clientes cadastrados
    print("TODOS OS CLIENTES CADASTRADOS")
    cursor.execute("SELECT * FROM clientes")
    todos_clientes = cursor.fetchall()
    
    for cliente in todos_clientes:
        print(cliente)

    # Consultar os pedidos com valor superior a R$ 30 (usando JOIN para trazer o nome do cliente)
    print("\nPEDIDOS COM VALOR SUPERIOR A R$ 30,00")
    consulta_pedidos = """
        SELECT 
            p.pedido_id,
            c.nome AS cliente,
            p.data_pedido,
            p.valor
        FROM pedidos p
        JOIN clientes c ON p.cliente_id = c.cliente_id
        WHERE p.valor > 30.00
    """
    cursor.execute(consulta_pedidos)

    #Recuperar os registros em lotes de dois utilizando fetchmany(2)
    lote_numero = 1
    while True:
        lote = cursor.fetchmany(2)
        if not lote:  # Se a lista vier vazia, todos os registros já foram lidos
            break
        
        print(f"\nLote {lote_numero}:")
        for pedido in lote:
            print(pedido)
        lote_numero += 1

    df_pedidos_filtrados = pd.read_sql_query(consulta_pedidos, conexao)
    display(df_pedidos_filtrados)
finally:
    # Fecha o cursor e a conexão com o banco
    cursor.close()
    conexao.close()


TODOS OS CLIENTES CADASTRADOS
(1, 'Ana', 'Recife')
(2, 'Bruno', 'Curitiba')
(3, 'Lia', 'Manaus')
(4, 'Beatriz', 'Aracaju')
(5, 'Patricia', 'Aracaju')

PEDIDOS COM VALOR SUPERIOR A R$ 30,00

Lote 1:
(1001, 'Ana', '2026-03-01', 45.9)
(1004, 'Lia', '2026-03-04', 75.0)

Lote 2:
(1005, 'Bruno', '2026-03-05', 32.25)
(1006, 'Ana', '2026-03-06', 78.9)

Lote 3:
(1007, 'Bruno', '2026-03-07', 108.0)
(1008, 'Ana', '2026-03-08', 32.5)

Lote 4:
(1010, 'Bruno', '2026-03-10', 999.25)


,pedido_id,cliente,data_pedido,valor
0,1001,Ana,2026-03-01,45.90
1,1004,Lia,2026-03-04,75.00
2,1005,Bruno,2026-03-05,32.25
3,1006,Ana,2026-03-06,78.90
4,1007,Bruno,2026-03-07,108.00
5,1008,Ana,2026-03-08,32.50
6,1010,Bruno,2026-03-10,999.25
